In [7]:
"""
Multiple Linear Regression with DBH Interactions 
Goal is to compate mulitple models
Standard model with DBH × Height interaction
Log transformation of response
Square root transformation of response
Log transformation of predictors
Main evaluation : RMSE
Secondary: MAE, R²
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
#from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [8]:
df = pd.read_csv("C:\\Users\\aiden\\OneDrive\\Desktop\\Main folder\\School\\Data Science\\411\\biomass_model_dataset.csv")
df.columns = ['Species', 'DBH', 'Height', 'Biomass']

print(f"\nDataset: {df.shape[0]} observations, {df.shape[1]} variables")



df_train = pd.read_csv("C:\\Users\\aiden\\OneDrive\\Desktop\\Main folder\\School\\Data Science\\411\\train_biomass.csv")
df_test = pd.read_csv("C:\\Users\\aiden\\OneDrive\\Desktop\\Main folder\\School\\Data Science\\411\\test_biomass.csv")
df_valid = pd.read_csv("C:\\Users\\aiden\\OneDrive\\Desktop\\Main folder\\School\\Data Science\\411\\validate_biomass.csv")

print(f"\nData Split:")
print(f"  Training:   {len(df_train)}")
print(f"  Validation: {len(df_valid)}")
print(f"  Test:       {len(df_test)}")


Dataset: 137079 observations, 4 variables

Data Split:
  Training:   45693
  Validation: 45693
  Test:       45693


In [9]:

def evaluate(y_true, y_pred):
    """Calculate RMSE (main), MAE, and R²."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

#Standard interaction

def features(data):
    return pd.DataFrame({
        'DO_BH': data['DO_BH'].values,
        'HT_TOT': data['HT_TOT'].values,
        'DO_BH_sq': data['DO_BH'].values ** 2,
        'DO_BH_x_HT': data['DO_BH'].values * data['HT_TOT'].values,
        'DO_BH2_x_HT': (data['DO_BH'].values ** 2) * data['HT_TOT'].values
    })

X_train_1 = features(df_train)
X_valid_1 = features(df_valid)
X_test_1 = features(df_test)
y_train = df_train['TT_DW_CRM'].values
y_valid = df_valid['TT_DW_CRM'].values
y_test = df_test['TT_DW_CRM'].values

model1 = LinearRegression()
model1.fit(X_train_1, y_train)
pred_valid_1 = model1.predict(X_valid_1)
rmse_1, mae_1, r2_1 = evaluate(y_valid, pred_valid_1)

print(f"  RMSE (main): {rmse_1:.2f} ")
print(f"  MAE:         {mae_1:.2f} ")
print(f"  R²:          {r2_1:.4f}")

print(f"Coef:")
print(f"  Intercept:     {model1.intercept_:.4f}")
for name, coef in zip(X_train_1.columns, model1.coef_):
    print(f"  {name:14s}: {coef:.6f}")

  RMSE (main): 706.27 
  MAE:         247.56 
  R²:          0.9986
Coef:
  Intercept:     -411.3880
  DO_BH         : 92.603481
  HT_TOT        : -3.127955
  DO_BH_sq      : -0.627679
  DO_BH_x_HT    : 0.197652
  DO_BH2_x_HT   : 0.047968


In [10]:
#Log in respect to response

def featureS(data):
    return pd.DataFrame({
        'DO_BH': data['DO_BH'].values,
        'HT_TOT': data['HT_TOT'].values,
        'DO_BH_x_HT': data['DO_BH'].values * data['HT_TOT'].values
    })

X_train_2 = featureS(df_train)
X_valid_2 = featureS(df_valid)
y_train_log = np.log(y_train)

model2 = LinearRegression()
model2.fit(X_train_2, y_train_log)
pred_valid_2_log = model2.predict(X_valid_2)
pred_valid_2 = np.exp(pred_valid_2_log)  # Back-transform
rmse_2, mae_2, r2_2 = evaluate(y_valid, pred_valid_2)

print(f"Performance:")
print(f"  RMSE (main): {rmse_2:.2f} kg")
print(f"  MAE:         {mae_2:.2f} kg")
print(f"  R²:          {r2_2:.4f}")

print(f"Coef(log):")
print(f"  Intercept:     {model2.intercept_:.6f}")
for name, coef in zip(X_train_2.columns, model2.coef_):
    print(f"  {name:14s}: {coef:.6f}")

Performance:
  RMSE (main): 342871.03 kg
  MAE:         3473.56 kg
  R²:          -337.7092
Coef(log):
  Intercept:     2.114678
  DO_BH         : 0.220330
  HT_TOT        : 0.034168
  DO_BH_x_HT    : -0.000762


In [11]:
y_train_cbrt = np.cbrt(y_train) 

model3 = LinearRegression()
model3.fit(X_train_2, y_train_cbrt)
pred_valid_3_cbrt = model3.predict(X_valid_2)
pred_valid_3 = pred_valid_3_cbrt ** 3  # Back-transform
rmse_3, mae_3, r2_3 = evaluate(y_valid, pred_valid_3)

print(f"Performance:")
print(f"  RMSE (main): {rmse_3:.2f} kg")
print(f"  MAE:         {mae_3:.2f} kg")
print(f"  R²:          {r2_3:.4f}")

print(f"Coef(cube root):")
print(f"  Intercept:     {model3.intercept_:.6f}")
for name, coef in zip(X_train_2.columns, model3.coef_):
    print(f"  {name:14s}: {coef:.6f}")

Performance:
  RMSE (main): 7534.54 kg
  MAE:         289.21 kg
  R²:          0.8364
Coef(cube root):
  Intercept:     -0.256830
  DO_BH         : 0.542097
  HT_TOT        : 0.054584
  DO_BH_x_HT    : -0.000666


In [12]:
#Log Transfomration for the predictoes

def featuresL(data):
    return pd.DataFrame({
        'log_DO_BH': np.log(data['DO_BH'].values),
        'log_HT_TOT': np.log(data['HT_TOT'].values)
    })

X_train_4a = featuresL(df_train)
X_valid_4a = featuresL(df_valid)

model4a = LinearRegression()
model4a.fit(X_train_4a, y_train_log)
pred_valid_4a_log = model4a.predict(X_valid_4a)
pred_valid_4a = np.exp(pred_valid_4a_log)
rmse_4a, mae_4a, r2_4a = evaluate(y_valid, pred_valid_4a)

print(f"Performance:")
print(f"  RMSE (main): {rmse_4a:.2f} kg")
print(f"  MAE:         {mae_4a:.2f} kg")
print(f"  R²:          {r2_4a:.4f}")

print(f"\Coef:")
print(f"  Int:     {model4a.intercept_:.6f}")
for name, coef in zip(X_train_4a.columns, model4a.coef_):
    print(f"  {name:14s}: {coef:.6f}")


Performance:
  RMSE (main): 11210.67 kg
  MAE:         429.72 kg
  R²:          0.6379
\Coef:
  Int:     -1.769385
  log_DO_BH     : 2.063562
  log_HT_TOT    : 0.797928


In [13]:
#Log with interaction
def featureLI(data):
    log_dbh = np.log(data['DO_BH'].values)
    log_ht = np.log(data['HT_TOT'].values)
    return pd.DataFrame({
        'log_DO_BH': log_dbh,
        'log_HT_TOT': log_ht,
        'log_DO_BH_x_log_HT': log_dbh * log_ht
    })

X_train_4b = featureLI(df_train)
X_valid_4b = featureLI(df_valid)

model4b = LinearRegression()
model4b.fit(X_train_4b, y_train_log)
pred_valid_4b_log = model4b.predict(X_valid_4b)
pred_valid_4b = np.exp(pred_valid_4b_log)
rmse_4b, mae_4b, r2_4b = evaluate(y_valid, pred_valid_4b)

print(f"Performance:")
print(f"  RMSE (main): {rmse_4b:.2f} kg")
print(f"  MAE:         {mae_4b:.2f} kg")
print(f"  R²:          {r2_4b:.4f}")

print(f"Coef:")
print(f"  Intercept:     {model4b.intercept_:.6f}")
for name, coef in zip(X_train_4b.columns, model4b.coef_):
    print(f"  {name:14s}: {coef:.6f}")

Performance:
  RMSE (main): 27691.32 kg
  MAE:         803.40 kg
  R²:          -1.2093
Coef:
  Intercept:     -1.283384
  log_DO_BH     : 1.787592
  log_HT_TOT    : 0.669504
  log_DO_BH_x_log_HT: 0.069997


In [14]:
#Log + Polynomial 

def featuesLP(data):
    log_dbh = np.log(data['DO_BH'].values)
    log_ht = np.log(data['HT_TOT'].values)
    return pd.DataFrame({
        'log_DO_BH': log_dbh,
        'log_HT_TOT': log_ht,
        'log_DO_BH_sq': log_dbh ** 2,
        'log_HT_TOT_sq': log_ht ** 2
    })

X_train_4c = featuesLP(df_train)
X_valid_4c = featuesLP(df_valid)

model4c = LinearRegression()
model4c.fit(X_train_4c, y_train_log)
pred_valid_4c_log = model4c.predict(X_valid_4c)
pred_valid_4c = np.exp(pred_valid_4c_log)
rmse_4c, mae_4c, r2_4c = evaluate(y_valid, pred_valid_4c)

print(f"Performance:")
print(f"  RMSE (main): {rmse_4c:.2f} kg")
print(f"  MAE:         {mae_4c:.2f} kg")
print(f"  R²:          {r2_4c:.4f}")

print(f"Coef:")
print(f"  Intercept:     {model4c.intercept_:.6f}")
for name, coef in zip(X_train_4c.columns, model4c.coef_):
    print(f"  {name:14s}: {coef:.6f}")

Performance:
  RMSE (main): 14082.93 kg
  MAE:         549.61 kg
  R²:          0.4286
Coef:
  Intercept:     0.556724
  log_DO_BH     : 2.228014
  log_HT_TOT    : -0.509301
  log_DO_BH_sq  : -0.040333
  log_HT_TOT_sq : 0.169741


In [15]:
#Log with BOTH interaction + poly 
def featuresLIP(data):
    log_dbh = np.log(data['DO_BH'].values)
    log_ht = np.log(data['HT_TOT'].values)
    return pd.DataFrame({
        'log_DO_BH': log_dbh,
        'log_HT_TOT': log_ht,
        'log_DO_BH_sq': log_dbh ** 2,
        'log_DO_BH_x_log_HT': log_dbh * log_ht
    })

X_train_4d = featuresLIP(df_train)
X_valid_4d = featuresLIP(df_valid)

model4d = LinearRegression()
model4d.fit(X_train_4d, y_train_log)
pred_valid_4d_log = model4d.predict(X_valid_4d)
pred_valid_4d = np.exp(pred_valid_4d_log)
rmse_4d, mae_4d, r2_4d = evaluate(y_valid, pred_valid_4d)

print(f"Performance:")
print(f"  RMSE (main): {rmse_4d:.2f} kg")
print(f"  MAE:         {mae_4d:.2f} kg")
print(f"  R²:          {r2_4d:.4f}")

print(f"Coef:")
print(f"  Intercept:     {model4d.intercept_:.6f}")
for name, coef in zip(X_train_4d.columns, model4d.coef_):
    print(f"  {name:14s}: {coef:.6f}")

Performance:
  RMSE (main): 12244.29 kg
  MAE:         515.54 kg
  R²:          0.5681
Coef:
  Intercept:     -0.247817
  log_DO_BH     : 1.597409
  log_HT_TOT    : 0.248889
  log_DO_BH_sq  : -0.141760
  log_DO_BH_x_log_HT: 0.269820


In [16]:
#Model COmparision
#lowk just made AI do this part for me
print(f"{'1. Standard with interaction':<50} {rmse_1:>10.2f} {mae_1:>10.2f} {r2_1:>8.4f}")
print(f"{'2. Log transform response':<50} {rmse_2:>10.2f} {mae_2:>10.2f} {r2_2:>8.4f}")
print(f"{'3. Cube root transform response':<50} {rmse_3:>10.2f} {mae_3:>10.2f} {r2_3:>8.4f}")
print(f"{'4. Log-Log (simple allometric)':<50} {rmse_4a:>10.2f} {mae_4a:>10.2f} {r2_4a:>8.4f}")
print(f"{'5. Log-Log with interaction':<50} {rmse_4b:>10.2f} {mae_4b:>10.2f} {r2_4b:>8.4f}")
print(f"{'6. Log-Log with polynomial':<50} {rmse_4c:>10.2f} {mae_4c:>10.2f} {r2_4c:>8.4f}")
print(f"{'7. Log-Log full (poly + interaction)':<50} {rmse_4d:>10.2f} {mae_4d:>10.2f} {r2_4d:>8.4f}")

# Find best model by RMSE
results = [
    (1, rmse_1, 'Standard with interaction'),
    (2, rmse_2, 'Log transform response'),
    (3, rmse_3, 'Cube root transform response'),
    (4, rmse_4a, 'Log-Log simple'),
    (5, rmse_4b, 'Log-Log with interaction'),
    (6, rmse_4c, 'Log-Log with polynomial'),
    (7, rmse_4d, 'Log-Log full')
]
best = min(results, key=lambda x: x[1])
best_model_num = best[0]
print(f"\n*** Best Model by RMSE: Model {best_model_num} ({best[2]}) ***")

1. Standard with interaction                           706.27     247.56   0.9986
2. Log transform response                           342871.03    3473.56 -337.7092
3. Cube root transform response                       7534.54     289.21   0.8364
4. Log-Log (simple allometric)                       11210.67     429.72   0.6379
5. Log-Log with interaction                          27691.32     803.40  -1.2093
6. Log-Log with polynomial                           14082.93     549.61   0.4286
7. Log-Log full (poly + interaction)                 12244.29     515.54   0.5681

*** Best Model by RMSE: Model 1 (Standard with interaction) ***


In [17]:
#This one evals on the test data, again didnt want to type all this so AI did it for me
#I NEED TO DOUBLE CHECK THIS DO NOT FINAL
#NOTTTT FINALL
df_train_final = pd.concat([df_train, df_valid])
y_train_final = df_train_final['TT_DW_CRM'].values

if best_model_num == 1:
    X_final = features(df_train_final)
    X_test_final = features(df_test)
    final_model = LinearRegression()
    final_model.fit(X_final, y_train_final)
    pred_test = final_model.predict(X_test_final)
    feature_names = X_final.columns
elif best_model_num == 2:
    X_final = featureS(df_train_final)
    X_test_final = featureS(df_test)
    final_model = LinearRegression()
    final_model.fit(X_final, np.log(y_train_final))
    pred_test = np.exp(final_model.predict(X_test_final))
    feature_names = X_final.columns
elif best_model_num == 3:
    X_final = featureS(df_train_final)
    X_test_final = featureS(df_test)
    final_model = LinearRegression()
    final_model.fit(X_final, np.cbrt(y_train_final))
    pred_test = final_model.predict(X_test_final) ** 3
    feature_names = X_final.columns
elif best_model_num == 4:
    X_final = featuresL(df_train_final)
    X_test_final = featuresL(df_test)
    final_model = LinearRegression()
    final_model.fit(X_final, np.log(y_train_final))
    pred_test = np.exp(final_model.predict(X_test_final))
    feature_names = X_final.columns
elif best_model_num == 5:
    X_final = featureLI(df_train_final)
    X_test_final = featureLI(df_test)
    final_model = LinearRegression()
    final_model.fit(X_final, np.log(y_train_final))
    pred_test = np.exp(final_model.predict(X_test_final))
    feature_names = X_final.columns
elif best_model_num == 6:
    X_final = featuesLP(df_train_final)
    X_test_final = featuesLP(df_test)
    final_model = LinearRegression()
    final_model.fit(X_final, np.log(y_train_final))
    pred_test = np.exp(final_model.predict(X_test_final))
    feature_names = X_final.columns
else:  # model 7
    X_final = featuresLIP(df_train_final)
    X_test_final = featuresLIP(df_test)
    final_model = LinearRegression()
    final_model.fit(X_final, np.log(y_train_final))
    pred_test = np.exp(final_model.predict(X_test_final))
    feature_names = X_final.columns

rmse_test, mae_test, r2_test = evaluate(y_test, pred_test)

print(f"\nTest Set Performance:")
print(f"  RMSE (main): {rmse_test:.2f} kg")
print(f"  MAE:         {mae_test:.2f} kg")
print(f"  R²:          {r2_test:.4f}")

print(f"\nFinal Model Coefficients:")
print(f"  Intercept:     {final_model.intercept_:.6f}")
for name, coef in zip(feature_names, final_model.coef_):
    print(f"  {name:14s}: {coef:.6f}")


Test Set Performance:
  RMSE (main): 689.83 kg
  MAE:         259.86 kg
  R²:          0.9985

Final Model Coefficients:
  Intercept:     -414.305477
  DO_BH         : 111.713445
  HT_TOT        : -5.552647
  DO_BH_sq      : -1.423557
  DO_BH_x_HT    : 0.252686
  DO_BH2_x_HT   : 0.049973


**Confidence Interval Stuff for the final model**

In [18]:
print("Confidence interval stuff to go here")




Confidence interval stuff to go here
